# Latin-Masking Pipeline (Two-Stage API)

This notebook runs the latin-masking pipeline using the `run_pipeline_stage1` and
`run_pipeline_stage2` API functions, which handle normalization, sentence splitting,
UDPipe POS tagging, adverb collection, -que splitting, and masking automatically.

## Pipeline Stages:
- **Stage 1**: Normalize → sentence-split → UDPipe → collect adverbs → save `common_adverbs.txt`
- **Review**: Manually review/edit `common_adverbs.txt` before proceeding
- **Stage 2**: -que split → UDPipe → mask

## Step 1: Setup and Imports

In [25]:
from pathlib import Path

from latin_masking import run_pipeline_stage1, run_pipeline_stage2
from latin_masking.types import MaskingConfig
import latin_masking.client as client


def find_project_root():
    current = Path.cwd().resolve()

    for folder in [current, *current.parents]:
        if (
            (folder / "data").is_dir()
            and (folder / "common_adverbs.txt").is_file()
        ):
            return folder

    raise FileNotFoundError("Lexical analysis root not found.")


BASE_DIR = find_project_root()

INPUT_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "1. De ordine" / "data_processed_ordine"

MODEL = "latin-evalatin24-240520"
ADVERB_THRESHOLD = 200

EXCLUDED_FILES = {
    "X4_certamina.txt",
    "X2_ecclesiastes",
    "X3_machabeis.txt",
}

txt_files = sorted([
    f for f in INPUT_DIR.glob("*.txt")
    if "_sentences" not in f.name
    and f.name not in EXCLUDED_FILES
])

stems = [f.stem for f in txt_files]
duplicates = [s for s in stems if stems.count(s) > 1]

if duplicates:
    print(f"\nWARNING: Found duplicate file stems: {set(duplicates)}")

config = MaskingConfig(
    output_dir=OUTPUT_DIR,
    model=MODEL,
    adverb_threshold=ADVERB_THRESHOLD,
    common_adverbs_path=BASE_DIR / "common_adverbs.txt",
    cache_dir=OUTPUT_DIR / "udpipe_cache",
    unsafe_certs_ok=True,
)

print(f"Project root: {BASE_DIR}")
print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

print(f"\nConfig: model={config.model}, threshold={config.adverb_threshold}")
print(f"  output_dir={config.output_dir}")
print(f"  cache_dir={config.cache_dir}")
print(f"  adverbs_path={config.common_adverbs_path}")

Project root: /Users/martinapaccara/Documents/Documenti/Università/POST-LAUREA/DOTTORATO/Lavoro/Digitale/Stilometria/hexameters/Lexical analysis
Input directory: /Users/martinapaccara/Documents/Documenti/Università/POST-LAUREA/DOTTORATO/Lavoro/Digitale/Stilometria/hexameters/Lexical analysis/data
Output directory: /Users/martinapaccara/Documents/Documenti/Università/POST-LAUREA/DOTTORATO/Lavoro/Digitale/Stilometria/hexameters/Lexical analysis/1. De ordine/data_processed_ordine

Config: model=latin-evalatin24-240520, threshold=200
  output_dir=/Users/martinapaccara/Documents/Documenti/Università/POST-LAUREA/DOTTORATO/Lavoro/Digitale/Stilometria/hexameters/Lexical analysis/1. De ordine/data_processed_ordine
  cache_dir=/Users/martinapaccara/Documents/Documenti/Università/POST-LAUREA/DOTTORATO/Lavoro/Digitale/Stilometria/hexameters/Lexical analysis/1. De ordine/data_processed_ordine/udpipe_cache
  adverbs_path=/Users/martinapaccara/Documents/Documenti/Università/POST-LAUREA/DOTTORAT

## Step 2: Run Stage 1 — Normalize, Split, UDPipe, Collect Adverbs

This runs the full stage 1 pipeline per file:
1. Read raw text
2. Sentence-split (with automatic preprocessing: UV/IJ normalization, macron removal, punctuation stripping)
3. UDPipe POS tagging (with caching)
4. Collect adverbs

After all files: aggregate → normalize counts → generate list → save `common_adverbs.txt`.

In [35]:
stage1_result = run_pipeline_stage1(txt_files, config=config, preserve_eol=True)

print(f"\n=== Stage 1 Summary ===")
print(f"Files processed: {len(stage1_result.sentences_per_file)}")
print(f"Total sentences: {sum(stage1_result.sentences_per_file.values())}")
print(f"Unique adverbs: {len(stage1_result.adverb_counts)}")
print(f"Adverbs saved to: {stage1_result.common_adverbs_path}")

→ AI_anticlaudianus.txt
✓ AI_anticlaudianus.txt: 1546 sentences (cached)
    229 adverbs collected
→ AeC_medicaminum.txt
✓ AeC_medicaminum.txt: 962 sentences (cached)
    217 adverbs collected
→ AeC_viaticus.txt
✓ AeC_viaticus.txt: 730 sentences (cached)
    116 adverbs collected
→ AeP_karolinus.txt
✓ AeP_karolinus.txt: 629 sentences (cached)
    295 adverbs collected
→ BC_contemptu.txt
✓ BC_contemptu.txt: 2720 sentences (cached)
    213 adverbs collected
→ B_judas.txt
✓ B_judas.txt: 401 sentences (cached)
    125 adverbs collected
→ DC_mathildis.txt
✓ DC_mathildis.txt: 1293 sentences (cached)
    305 adverbs collected
→ E_biblia.txt
✓ E_biblia.txt: 511 sentences (cached)
    203 adverbs collected
→ FB_nuptis.txt
✓ FB_nuptis.txt: 2657 sentences (cached)
    295 adverbs collected
→ GAp_roberti.txt
✓ GAp_roberti.txt: 1375 sentences (cached)
    222 adverbs collected
→ GAr_philippis.txt
✓ GAr_philippis.txt: 1796 sentences (cached)
    373 adverbs collected
→ GC_alexandreis.txt
✓ GC_alexan

In [28]:
for path in txt_files:
    data = path.read_bytes()

    try:
        data.decode("utf-8")
    except UnicodeDecodeError as e:
        pos = e.start

        line_number = data[:pos].count(b"\n") + 1

        print("=" * 70)
        print(f"FILE: {path}")
        print(f"BYTE POSITION: {pos}")
        print(f"LINE: {line_number}")
        print(f"BYTE: {data[pos:pos+1]!r} = 0x{data[pos]:02X}")

        start = max(0, pos - 100)
        end = min(len(data), pos + 100)

        print("\nCONTEXT:")
        print(data[start:end].decode("utf-8", errors="replace"))

## Step 3: Review Common Adverbs

Stage 1 has written `common_adverbs.txt`. Review and edit it if needed before
proceeding to Stage 2. The adverb list is used both for masking and to protect
common adverbs from -que splitting.

In [30]:
print(
    f"\nReview {config.common_adverbs_path} and edit if needed, then proceed to Stage 2."
)


Review /Users/martinapaccara/Documents/Documenti/Università/POST-LAUREA/DOTTORATO/Lavoro/Digitale/Stilometria/hexameters/Lexical analysis/common_adverbs.txt and edit if needed, then proceed to Stage 2.


## Step 4: Run Stage 2 — -que Split, UDPipe, Mask

This runs the full stage 2 pipeline per file:
1. Read sentence-split files from Stage 1
2. Load common adverbs; add -que adverbs to effective blacklist
3. Apply -que splitting → write `*.quesplit.txt`
4. UDPipe POS tagging (with caching)
5. Two-pass masking → write `*.quesplit.masked.txt`

In [57]:
client.process_text.__kwdefaults__["service_url"] = \
    "https://lindat.mff.cuni.cz/services/udpipe/api"

In [59]:
stage2_result = run_pipeline_stage2(txt_files, config=config)

print(f"\n=== Stage 2 Summary ===")
print(f"Output files: {len(stage2_result.output_files)}")
print(f"Sentences processed: {stage2_result.sentences_processed}")
print(f"Cache hits: {stage2_result.cache_hits}")
for f in stage2_result.output_files:
    print(f"  → {f.name}")

→ AI_anticlaudianus.txt
✓ AI_anticlaudianus.txt: 937 -que splits (cached)
    1546 masked sentences -> AI_anticlaudianus_sentences.quesplit.masked.txt
→ AeC_medicaminum.txt
✓ AeC_medicaminum.txt: 683 -que splits (cached)
    962 masked sentences -> AeC_medicaminum_sentences.quesplit.masked.txt
→ AeC_viaticus.txt
✓ AeC_viaticus.txt: 189 -que splits (cached)
    730 masked sentences -> AeC_viaticus_sentences.quesplit.masked.txt
→ AeP_karolinus.txt
✓ AeP_karolinus.txt: 407 -que splits (cached)
    629 masked sentences -> AeP_karolinus_sentences.quesplit.masked.txt
→ BC_contemptu.txt
✓ BC_contemptu.txt: 658 -que splits (cached)
    2720 masked sentences -> BC_contemptu_sentences.quesplit.masked.txt
→ B_judas.txt
✓ B_judas.txt: 132 -que splits (cached)
    401 masked sentences -> B_judas_sentences.quesplit.masked.txt
→ DC_mathildis.txt
✓ DC_mathildis.txt: 429 -que splits (cached)
    1293 masked sentences -> DC_mathildis_sentences.quesplit.masked.txt
→ E_biblia.txt
✓ E_biblia.txt: 380 -que 

## Step 5: View Results

Compare original, sentences, and masked output for the first file.

In [61]:
# Show example output
example_file = txt_files[0]  # First input file
print(f"=== Original ({example_file.name}) ===")
with open(example_file, "r", encoding="utf-8") as f:
    original = f.read()
print(original[:1000] + "..." if len(original) > 1000 else original)

sentences_file = config.output_dir / f"{example_file.stem}_sentences.txt"
print(f"\n=== Sentences ({sentences_file.name}) ===")
with open(sentences_file, "r", encoding="utf-8") as f:
    sentences = f.readlines()
for i, sent in enumerate(sentences[:10]):
    print(f"{i+1}. {sent.strip()}")
print(f"... ({len(sentences)} total sentences)")

masked_file = config.output_dir / f"{example_file.stem}_sentences.quesplit.masked.txt"
print(f"\n=== Masked ({masked_file.name}) ===")
with open(masked_file, "r", encoding="utf-8") as f:
    masked = f.readlines()
for i, sent in enumerate(masked[:10]):
    print(f"{i+1}. {sent.strip()}")
print(f"... ({len(masked)} total sentences)")

=== Original (AI_anticlaudianus.txt) ===
Auctoris mendico stylum, phalerasque poetae,
Ne mea segnitiae Clio deiecta senescat:
Ne iaceat calamus, scabra rubigine torpens.
Scribendi novitate vetus iuvenescere charta
Gaudet, et antiquas cupiens exire latebras,
Ridet, et in tenui lascivit arundine musa.
Fonte tuo sic, Phoebe, tuum perfunde poetam,
Ut compluta tuo mens arida flumine, germen
Donet, et in fructum concludat germinis usum.
Ut sibi iuncta magis Naturae dona resultent,
Et proprium donet donis mistura pavorem,
Solers Naturae studium, quae singula sparsim
Munera contulerat aliis, concludit in unum;
Cudit opus, per quod operi concluditur omni.
Pristina sic operum peccata repensat in uno,
Ut quod deliquit alias, compenset in isto.
Supplicat huic operi, famulans opus omne decoris;
Et tanta cupiens vestiri dote favoris,
Incudis deposcit opem; sed fessa laborat
Incus, quae tantos vires expendit in usus,
Ultra sese posse studet Natura, suumque
Supra se metitur opus, sic vincere fertur
Ar